In [10]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)


Token loaded: True


In [11]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git

Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 53 (delta 20), reused 31 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 322.68 KiB | 3.10 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [12]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [13]:
!git config --global credential.helper store

In [14]:

import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")


GitHub authentication configured.


In [15]:
!git fetch origin
!git switch nehna



Already on 'nehna'
Your branch is up to date with 'origin/nehna'.


In [16]:
!git status

On branch nehna
Your branch is up to date with 'origin/nehna'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	multilingual-rag-research/

nothing added to commit but untracked files present (use "git add" to track)


In [18]:
import json

with open("data/corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

with open("data/questions.json", encoding="utf-8") as f:
    questions = json.load(f)

with open("results/bm25_top10.json", encoding="utf-8") as f:
    bm25_results = json.load(f)

with open("results/dense_top10.json", encoding="utf-8") as f:
    dense_results = json.load(f)

print("Corpus:", len(corpus))
print("Questions:", len(questions))
print("BM25 results:", len(bm25_results))
print("Dense results:", len(dense_results))

Corpus: 1000
Questions: 100
BM25 results: 100
Dense results: 100


In [19]:
def recall_at_k(questions, results, k):
    hits = 0

    for q in questions:
        retrieved = results[q["question_id"]][:k]

        if q["gold_passage_id"] in retrieved:
            hits += 1

    return hits / len(questions)

In [20]:
def mrr(questions, results):
    total = 0.0

    for q in questions:
        retrieved = results[q["question_id"]]

        if q["gold_passage_id"] in retrieved:
            rank = retrieved.index(q["gold_passage_id"]) + 1
            total += 1.0 / rank

    return total / len(questions)

In [21]:
def reciprocal_rank_fusion(
    bm25_ranked_ids,
    dense_ranked_ids,
    k=60
):
    scores = {}

    for rank, passage_id in enumerate(
        bm25_ranked_ids,
        start=1
    ):
        scores[passage_id] = (
            scores.get(passage_id, 0.0)
            + 1.0 / (k + rank)
        )

    for rank, passage_id in enumerate(
        dense_ranked_ids,
        start=1
    ):
        scores[passage_id] = (
            scores.get(passage_id, 0.0)
            + 1.0 / (k + rank)
        )

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        passage_id
        for passage_id, score in ranked
    ]

In [22]:
fake_bm25 = [10, 20, 30]
fake_dense = [20, 10, 40]

result = reciprocal_rank_fusion(
    fake_bm25,
    fake_dense,
    k=60
)

print(result)

[10, 20, 30, 40]


In [23]:
hybrid_results = {}

for q in questions:
    qid = q["question_id"]

    hybrid_results[qid] = reciprocal_rank_fusion(
        bm25_results[qid],
        dense_results[qid],
        k=60
    )

In [24]:
for k in [1, 3, 5, 10]:
    print(
        f"Hybrid Recall@{k}:",
        recall_at_k(
            questions,
            hybrid_results,
            k
        )
    )

print(
    "Hybrid MRR:",
    mrr(questions, hybrid_results)
)

Hybrid Recall@1: 0.84
Hybrid Recall@3: 0.93
Hybrid Recall@5: 0.95
Hybrid Recall@10: 0.97
Hybrid MRR: 0.8919404761904762


In [25]:
print(
    f"{'k':<5}"
    f"{'BM25':<10}"
    f"{'Dense':<10}"
    f"{'Hybrid':<10}"
)

for k in [1, 3, 5, 10]:

    b = recall_at_k(
        questions,
        bm25_results,
        k
    )

    d = recall_at_k(
        questions,
        dense_results,
        k
    )

    h = recall_at_k(
        questions,
        hybrid_results,
        k
    )

    print(
        f"{k:<5}"
        f"{b:<10.3f}"
        f"{d:<10.3f}"
        f"{h:<10.3f}"
    )

print()

print(
    "BM25 MRR:  ",
    mrr(questions, bm25_results)
)

print(
    "Dense MRR: ",
    mrr(questions, dense_results)
)

print(
    "Hybrid MRR:",
    mrr(questions, hybrid_results)
)

k    BM25      Dense     Hybrid    
1    0.790     0.760     0.840     
3    0.880     0.950     0.930     
5    0.910     0.950     0.950     
10   0.950     0.970     0.970     

BM25 MRR:   0.8459682539682539
Dense MRR:  0.846111111111111
Hybrid MRR: 0.8919404761904762


In [26]:
for rrf_k in [10, 60, 100]:

    trial_results = {}

    for q in questions:

        qid = q["question_id"]

        trial_results[qid] = reciprocal_rank_fusion(
            bm25_results[qid],
            dense_results[qid],
            k=rrf_k
        )

    print(
        f"RRF k={rrf_k}: "
        f"Recall@5 = "
        f"{recall_at_k(questions, trial_results, 5):.3f}, "
        f"MRR = "
        f"{mrr(questions, trial_results):.3f}"
    )

RRF k=10: Recall@5 = 0.950, MRR = 0.892
RRF k=60: Recall@5 = 0.950, MRR = 0.892
RRF k=100: Recall@5 = 0.950, MRR = 0.892


In [27]:
rescued = []

for q in questions:

    qid = q["question_id"]

    b_hit = (
        q["gold_passage_id"]
        in bm25_results[qid][:5]
    )

    d_hit = (
        q["gold_passage_id"]
        in dense_results[qid][:5]
    )

    h_hit = (
        q["gold_passage_id"]
        in hybrid_results[qid][:5]
    )

    if not b_hit and not d_hit and h_hit:
        rescued.append(q)

print(
    "Questions rescued by hybrid:",
    len(rescued)
)

Questions rescued by hybrid: 0


In [28]:
for q in rescued[:3]:

    print("Question:", q["question"])

    print(
        "Gold passage:",
        corpus[q["gold_passage_id"]]["text"][:300]
    )

    print("-" * 50)

In [29]:
for q in questions:

    qid = q["question_id"]

    b_hit = (
        q["gold_passage_id"]
        in bm25_results[qid][:5]
    )

    d_hit = (
        q["gold_passage_id"]
        in dense_results[qid][:5]
    )

    h_hit = (
        q["gold_passage_id"]
        in hybrid_results[qid][:5]
    )

    if (b_hit or d_hit) and not h_hit:
        print(
            "Hybrid REGRESSED:",
            q["question"]
        )

Hybrid REGRESSED: What is th elast name of the player who was the Super Bowl 50 winner's leading rusher?
Hybrid REGRESSED: What did giving money to the church absolve the giver from?


In [30]:
with open(
    "results/hybrid_top10.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        hybrid_results,
        f,
        indent=2
    )

In [31]:
import csv
import os

os.makedirs("results", exist_ok=True)

with open(
    "results/retrieval_comparison.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "k",
        "bm25_recall",
        "dense_recall",
        "hybrid_recall"
    ])

    for k in [1, 3, 5, 10]:

        writer.writerow([
            k,

            recall_at_k(
                questions,
                bm25_results,
                k
            ),

            recall_at_k(
                questions,
                dense_results,
                k
            ),

            recall_at_k(
                questions,
                hybrid_results,
                k
            )
        ])